In [2]:
print("GPU Information:")

!nvidia-smi

GPU Information:
Tue Mar 24 05:42:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   36C    P0             66W /  400W |    5806MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+------------------------------

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

SYNAPSE = "/content/drive/MyDrive/synapse"
SHARD_DIR = os.path.join(SYNAPSE, "token_shards")
TOKENIZER_PATH = os.path.join(SYNAPSE, "tokenizer_out", "tokenizer.json")

# Copy tokenizer locally for fast access
!cp "{TOKENIZER_PATH}" "/content/tokenizer.json"

# Verify shards exist
if not os.path.isdir(SHARD_DIR):
    raise FileNotFoundError(f"Token shards not found at {SHARD_DIR}. Run tokenizer_pipeline.ipynb first.")

shard_files = sorted([f for f in os.listdir(SHARD_DIR) if f.startswith("shard_") and f.endswith(".bin")])
print(f"Found {len(shard_files)} shards in {SHARD_DIR}")

In [ ]:
#!/usr/bin/env python3
import os

# ==================== 0. MEMORY OPTIMIZATIONS (MUST BE FIRST) ====================
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import json
import hashlib
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torch.utils.checkpoint import checkpoint
from tqdm import tqdm
from collections import defaultdict
import math
import time
import shutil
import datetime
import logging

torch._logging.set_logs(inductor=logging.WARNING, dynamic=logging.WARNING)
logging.getLogger("torch._inductor").setLevel(logging.WARNING)

# ==================== 1. SETUP ====================
def ensure_drive_mounted():
    if os.path.isdir("/content/drive/MyDrive"):
        return
    try:
        from google.colab import drive
        print("Mounting Google Drive...")
        drive.mount("/content/drive", force_remount=False)
    except ImportError:
        print("Drive mount failed.")

ensure_drive_mounted()

# ==================== MANIFEST HELPERS ====================
def file_hash(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while chunk := f.read(8192):
            h.update(chunk)
    return h.hexdigest()

def file_info(path):
    if not os.path.exists(path):
        return {"path": path, "exists": False}
    return {
        "path": path,
        "size_mb": round(os.path.getsize(path) / 1024 / 1024, 2),
        "sha256": file_hash(path),
    }

# ==================== 2. CONFIGURATION ====================
SYNAPSE = "/content/drive/MyDrive/synapse"
SHARD_DIR = os.path.join(SYNAPSE, "token_shards")
CHECKPOINT_DIR = os.path.join(SYNAPSE, "checkpoints")
MANIFEST_DIR = os.path.join(SYNAPSE, "manifests")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(MANIFEST_DIR, exist_ok=True)

CHECKPOINT_NAME = "synapse_2b_d2560_l28.pth"
CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, CHECKPOINT_NAME)

# -- Archive old checkpoint if it exists --
if os.path.exists(CHECKPOINT_PATH):
    mod_time = os.path.getmtime(CHECKPOINT_PATH)
    date_str = datetime.datetime.fromtimestamp(mod_time).strftime("%Y%m%d_%H%M%S")
    archived_name = CHECKPOINT_NAME.replace(".pth", f"_{date_str}.pth")
    archived_path = os.path.join(CHECKPOINT_DIR, archived_name)
    if not os.path.exists(archived_path):
        shutil.copy2(CHECKPOINT_PATH, archived_path)
        print(f"Archived old checkpoint as: {archived_name}")

# -- MODEL ARCHITECTURE (Shape D, ~2.1B params) --
BLOCK_SIZE      = 2048
STRIDE          = BLOCK_SIZE
EMBED_DIM       = 2560
NUM_LAYERS      = 28
NUM_HEADS       = 20            # head_dim = 128
NUM_KV_HEADS    = 4             # GQA, group size = 5
FF_HIDDEN_DIM   = 6912          # ~8/3 * EMBED_DIM, multiple of 128
DROPOUT         = 0.1
ROPE_BASE       = 10000.0
RMSNORM_EPS     = 1e-5
GRAD_CHECKPOINT = True          # activation checkpointing every block

# -- TRAINING HYPERPARAMETERS --
BATCH_SIZE       = 4
GRAD_ACCUM_STEPS = 64           # effective batch = 256 sequences = 524K tokens/step
EPOCHS           = 1
MAX_LR           = 2e-4
MIN_LR           = 2e-5
WEIGHT_DECAY     = 0.1
WARMUP_STEPS     = 4000
BETAS            = (0.9, 0.95)  # Llama-style
GRAD_CLIP        = 1.0

# -- DATA SELECTION --
# Chinchilla-optimal for 2.1B is ~42B tokens (20:1). You have ~130B available;
# bump this to 80B-130B for a more overtrained / better model if wall-clock allows.
MAX_TOKENS = 42_000_000_000

DATA_MIX = {
    "data_wikipedia": 0.40,
    "data_c4":        0.20,
    "data_code":      0.15,
    "data_math":      0.10,
    "data_books":     0.05,
    "data_fadedpage": 0.03,
    "data_tech":      0.02,
    "data_adult":     0.02,
    "data_religions": 0.03,
}

# -- EVAL --
EVAL_FRACTION_PER_SOURCE = 0.02   # hold out 2% of shards per source for eval
EVAL_EVERY_STEPS         = 500    # run eval every N optimizer steps
EVAL_BATCHES             = 32     # batches per eval pass
EVAL_SEED                = 1337

# -- MID-EPOCH CHECKPOINT --
SAVE_EVERY_N_SHARDS = 50

# Hardware Settings
torch.set_float32_matmul_precision('high')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == 'cuda':
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name} | VRAM: {props.total_memory / 1024**3:.1f} GB")

# ==================== 3. MODEL DEFINITION ====================

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        # Cast to fp32 for the norm computation; cast back to input dtype.
        in_dtype = x.dtype
        x = x.float()
        rms = x.pow(2).mean(-1, keepdim=True).add(self.eps).rsqrt()
        return (x * rms).to(in_dtype) * self.weight

def precompute_rope(head_dim, max_seq_len, base, device, dtype=torch.float32):
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2, device=device, dtype=dtype) / head_dim))
    t = torch.arange(max_seq_len, device=device, dtype=dtype)
    freqs = torch.outer(t, inv_freq)            # (T, head_dim/2)
    cos = torch.cat([freqs.cos(), freqs.cos()], dim=-1)  # (T, head_dim)
    sin = torch.cat([freqs.sin(), freqs.sin()], dim=-1)
    return cos, sin

def apply_rope(x, cos, sin):
    # x: (B, n_heads, T, head_dim); cos/sin: (T, head_dim)
    T = x.size(-2)
    cos = cos[:T].unsqueeze(0).unsqueeze(0)
    sin = sin[:T].unsqueeze(0).unsqueeze(0)
    half = x.size(-1) // 2
    x1, x2 = x[..., :half], x[..., half:]
    rotated = torch.cat([-x2, x1], dim=-1)
    return (x * cos) + (rotated * sin)

class CausalGQA(nn.Module):
    def __init__(self, embed_dim, num_heads, num_kv_heads, dropout):
        super().__init__()
        assert embed_dim % num_heads == 0
        assert num_heads % num_kv_heads == 0
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.head_dim = embed_dim // num_heads
        self.n_rep = num_heads // num_kv_heads
        kv_dim = num_kv_heads * self.head_dim
        self.q_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.k_proj = nn.Linear(embed_dim, kv_dim, bias=False)
        self.v_proj = nn.Linear(embed_dim, kv_dim, bias=False)
        self.o_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.dropout = dropout
    def forward(self, x, cos, sin):
        B, T, C = x.size()
        q = self.q_proj(x).view(B, T, self.num_heads,    self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.num_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.num_kv_heads, self.head_dim).transpose(1, 2)
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)
        # Manual GQA expansion (PyTorch-version-agnostic; SDPA's enable_gqa requires 2.5+).
        if self.n_rep > 1:
            k = k.repeat_interleave(self.n_rep, dim=1)
            v = v.repeat_interleave(self.n_rep, dim=1)
        y = F.scaled_dot_product_attention(
            q, k, v,
            is_causal=True,
            dropout_p=self.dropout if self.training else 0.0,
        )
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.o_proj(y)

class SwiGLU(nn.Module):
    def __init__(self, embed_dim, hidden_dim):
        super().__init__()
        self.w1 = nn.Linear(embed_dim, hidden_dim * 2, bias=False)
        self.w2 = nn.Linear(hidden_dim, embed_dim,     bias=False)
    def forward(self, x):
        x1, x2 = self.w1(x).chunk(2, dim=-1)
        return self.w2(F.silu(x1) * x2)

class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, num_kv_heads, ff_hidden_dim, dropout, rms_eps):
        super().__init__()
        self.norm1 = RMSNorm(embed_dim, eps=rms_eps)
        self.attn  = CausalGQA(embed_dim, num_heads, num_kv_heads, dropout)
        self.norm2 = RMSNorm(embed_dim, eps=rms_eps)
        self.ff    = SwiGLU(embed_dim, ff_hidden_dim)
        self.dropout = dropout
    def forward(self, x, cos, sin):
        x = x + F.dropout(self.attn(self.norm1(x), cos, sin), p=self.dropout, training=self.training)
        x = x + F.dropout(self.ff(self.norm2(x)),             p=self.dropout, training=self.training)
        return x

class SynapseGPT(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, EMBED_DIM)
        self.blocks = nn.ModuleList([
            TransformerBlock(EMBED_DIM, NUM_HEADS, NUM_KV_HEADS, FF_HIDDEN_DIM, DROPOUT, RMSNORM_EPS)
            for _ in range(NUM_LAYERS)
        ])
        self.final_norm = RMSNorm(EMBED_DIM, eps=RMSNORM_EPS)
        self.lm_head = nn.Linear(EMBED_DIM, vocab_size, bias=False)
        self.token_embedding.weight = self.lm_head.weight   # tied
        # RoPE tables (registered as buffers, not parameters)
        head_dim = EMBED_DIM // NUM_HEADS
        cos, sin = precompute_rope(head_dim, BLOCK_SIZE, ROPE_BASE, device="cpu")
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
        self.gradient_checkpointing = GRAD_CHECKPOINT
        self.apply(self._init_weights)
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None: torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
    def forward(self, idx):
        x = self.token_embedding(idx)
        cos, sin = self.rope_cos, self.rope_sin
        for block in self.blocks:
            if self.gradient_checkpointing and self.training:
                x = checkpoint(block, x, cos, sin, use_reentrant=False)
            else:
                x = block(x, cos, sin)
        return self.lm_head(self.final_norm(x))

# ==================== 4. SHARDED DATASET ====================
class ShardDataset(Dataset):
    def __init__(self, tokens, block_size, stride):
        self.tokens = tokens
        self.block_size = block_size
        self.stride = stride
        self.length = max(0, (len(tokens) - block_size - 1) // stride)
    def __len__(self): return self.length
    def __getitem__(self, idx):
        s = idx * self.stride
        if s + self.block_size + 1 > len(self.tokens):
            s = len(self.tokens) - self.block_size - 1
        chunk = self.tokens[s : s + self.block_size + 1]
        return torch.from_numpy(chunk[:-1].copy()), torch.from_numpy(chunk[1:].copy())

# ==================== 5. LOAD META & MODEL ====================
META_PATH = os.path.join(SHARD_DIR, "meta.json")
with open(META_PATH, "r") as f:
    meta = json.load(f)
VOCAB_SIZE_RAW = int(meta["vocab_size"])
VOCAB_SIZE = VOCAB_SIZE_RAW
if VOCAB_SIZE % 64 != 0:
    VOCAB_SIZE = ((VOCAB_SIZE + 63) // 64) * 64
    print(f"Vocab padded to {VOCAB_SIZE} (Tensor Core Optimized)")

SHARD_DTYPE = np.dtype(meta.get("shard_dtype", "uint16"))
assert VOCAB_SIZE_RAW <= np.iinfo(SHARD_DTYPE).max + 1, (
    f"vocab_size {VOCAB_SIZE_RAW} exceeds {SHARD_DTYPE} range -- shards would overflow"
)
print(f"shard_dtype: {SHARD_DTYPE} | vocab raw: {VOCAB_SIZE_RAW} | padded: {VOCAB_SIZE}")

current_tok_id = meta.get("tokenization_id")
print(f"tokenization_id: {current_tok_id}")

model = SynapseGPT(VOCAB_SIZE).to(device)

n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: {n_params/1e9:.3f}B params ({n_trainable/1e9:.3f}B trainable)")

# --- CHECKPOINT LOADING ---
resume_path = None
if os.path.exists(CHECKPOINT_PATH):
    train_manifest_path = os.path.join(MANIFEST_DIR, "training_latest.json")
    safe_to_load = True
    if os.path.exists(train_manifest_path) and current_tok_id:
        with open(train_manifest_path) as f:
            train_manifest = json.load(f)
        ckpt_tok_id = train_manifest.get("tokenization_id")
        if ckpt_tok_id and ckpt_tok_id != current_tok_id:
            print(f"TOKENIZER MISMATCH — checkpoint: {ckpt_tok_id}, current: {current_tok_id}")
            print(f"  Training from scratch.")
            safe_to_load = False
        else:
            print(f"Tokenizer match confirmed.")
    if safe_to_load:
        print(f"Resuming from: {CHECKPOINT_PATH}")
        resume_path = CHECKPOINT_PATH
        state_dict = torch.load(CHECKPOINT_PATH, map_location=device)
        new_state = {k.replace("_orig_mod.", ""): v for k, v in state_dict.items()}
        model_state = model.state_dict()
        safe_state = {k: v for k, v in new_state.items() if k in model_state and v.shape == model_state[k].shape}
        model.load_state_dict(safe_state, strict=False)
        print(f"State loaded: {len(safe_state)} layers matched.")
else:
    print(f"No checkpoint found — training from scratch.")

print("Compiling model...")
model = torch.compile(model)

# Optimizer
param_dict = {pn: p for pn, p in model.named_parameters() if p.requires_grad}
decay_params = [p for n, p in param_dict.items() if p.dim() >= 2]
nodecay_params = [p for n, p in param_dict.items() if p.dim() < 2]
optim_groups = [
    {'params': decay_params,   'weight_decay': WEIGHT_DECAY},
    {'params': nodecay_params, 'weight_decay': 0.0},
]
optimizer = optim.AdamW(optim_groups, lr=MAX_LR, betas=BETAS, fused=True)
print(f"Optimizer: AdamW(fused) lr={MAX_LR} betas={BETAS} wd={WEIGHT_DECAY}")
print(f"  decay groups: {sum(p.numel() for p in decay_params)/1e6:.1f}M params, "
      f"nodecay: {sum(p.numel() for p in nodecay_params)/1e6:.3f}M params")

# ==================== 6. SELECT SHARDS (TRAIN + EVAL SPLIT) ====================
with open(os.path.join(SHARD_DIR, "shard_manifest.json"), "r") as f:
    shard_manifest = json.load(f)

all_shards = shard_manifest["shards"]

def get_source_name(shard_entry):
    source = shard_entry.get("source", "")
    parts = source.replace("\\", "/").split("/")
    for p in parts:
        if p.startswith("data_"):
            return p
    return "other"

shards_by_source = defaultdict(list)
for s in all_shards:
    shards_by_source[get_source_name(s)].append(s)

print(f"\nAvailable data sources:")
for src, shards in sorted(shards_by_source.items()):
    src_tokens = sum(s["tokens"] for s in shards)
    print(f"  {src}: {len(shards)} shards, {src_tokens:,} tokens")

# Hold out a deterministic eval slice per source BEFORE selecting train shards.
eval_rng = random.Random(EVAL_SEED)
eval_shards = []
train_pool_by_source = {}
for src, shards in shards_by_source.items():
    pool = shards.copy()
    eval_rng.shuffle(pool)
    n_eval = max(1, int(len(pool) * EVAL_FRACTION_PER_SOURCE)) if len(pool) > 1 else 0
    eval_shards.extend(pool[:n_eval])
    train_pool_by_source[src] = pool[n_eval:]
print(f"\nHeld out {len(eval_shards)} eval shards "
      f"({sum(s['tokens'] for s in eval_shards):,} tokens)")

random.seed(42)
selected_shards = []
if DATA_MIX is None:
    pool = [s for ss in train_pool_by_source.values() for s in ss]
    random.shuffle(pool)
    budget = MAX_TOKENS if MAX_TOKENS else float('inf')
    running = 0
    for s in pool:
        if running >= budget: break
        selected_shards.append(s)
        running += s["tokens"]
else:
    budget = MAX_TOKENS if MAX_TOKENS else sum(s["tokens"] for ss in train_pool_by_source.values() for s in ss)
    for source, weight in DATA_MIX.items():
        source_budget = int(budget * weight)
        available = train_pool_by_source.get(source, [])
        if not available:
            print(f"  WARNING: {source} not found in shards, skipping")
            continue
        random.shuffle(available)
        running = 0
        for s in available:
            if running >= source_budget: break
            selected_shards.append(s)
            running += s["tokens"]

selected_tokens = sum(s["tokens"] for s in selected_shards)

print(f"\nSelected for training:")
selected_by_source = defaultdict(lambda: {"shards": 0, "tokens": 0})
for s in selected_shards:
    src = get_source_name(s)
    selected_by_source[src]["shards"] += 1
    selected_by_source[src]["tokens"] += s["tokens"]
for src, info in sorted(selected_by_source.items()):
    total_available = sum(s["tokens"] for s in shards_by_source.get(src, []))
    pct = info["tokens"] / total_available * 100 if total_available else 0
    print(f"  {src}: {info['shards']} shards, {info['tokens']:,} tokens ({pct:.0f}% of available)")
print(f"\n  TOTAL: {len(selected_shards)} shards, {selected_tokens:,} tokens")

# ==================== 7. COMPUTE TOTAL STEPS ====================
samples_approx = selected_tokens // BLOCK_SIZE
batches_total = samples_approx // BATCH_SIZE
total_steps = (batches_total * EPOCHS) // GRAD_ACCUM_STEPS

print(f"  Effective Batch Size: {BATCH_SIZE * GRAD_ACCUM_STEPS} sequences "
      f"({BATCH_SIZE * GRAD_ACCUM_STEPS * BLOCK_SIZE:,} tokens/step)")
print(f"  Total optimizer steps: {total_steps:,} | Warmup: {WARMUP_STEPS}")
print(f"  Mid-epoch save every {SAVE_EVERY_N_SHARDS} shards | Eval every {EVAL_EVERY_STEPS} steps")

# ==================== 8. EVAL HELPER ====================
@torch.no_grad()
def run_eval(model, eval_shards, n_batches):
    model.eval()
    losses = []
    rng = random.Random(EVAL_SEED)
    pool = eval_shards.copy()
    rng.shuffle(pool)
    seen = 0
    for shard_info in pool:
        shard_path = os.path.join(SHARD_DIR, shard_info["shard"])
        try:
            tokens = np.fromfile(shard_path, dtype=SHARD_DTYPE).astype(np.int64)
        except FileNotFoundError:
            continue
        ds = ShardDataset(tokens, BLOCK_SIZE, STRIDE)
        if len(ds) == 0:
            continue
        loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
        for xb, yb in loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)
            with torch.amp.autocast("cuda", dtype=torch.bfloat16):
                logits = model(xb)
                loss = F.cross_entropy(logits.view(-1, VOCAB_SIZE), yb.view(-1))
            losses.append(loss.item())
            seen += 1
            if seen >= n_batches:
                break
        del tokens, ds, loader
        if seen >= n_batches:
            break
    model.train()
    if not losses:
        return float('nan')
    return sum(losses) / len(losses)

# ==================== 9. TRAINING LOOP ====================
def get_lr(it, total_it):
    if it < WARMUP_STEPS: return MAX_LR * it / WARMUP_STEPS
    decay_ratio = (it - WARMUP_STEPS) / max(1, (total_it - WARMUP_STEPS))
    coeff = 0.5 * (1.0 + math.cos(math.pi * min(1.0, decay_ratio)))
    return MIN_LR + coeff * (MAX_LR - MIN_LR)

curr_step = 0
final_loss = None
last_grad_norm = None
last_eval_loss = None
eval_history = []
train_start = time.time()

print("\nSTARTING TRAINING (BFloat16, GQA, RoPE, RMSNorm, grad-ckpt)\n")

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_shards = selected_shards.copy()
    random.shuffle(epoch_shards)

    for shard_idx, shard_info in enumerate(epoch_shards):
        shard_path = os.path.join(SHARD_DIR, shard_info["shard"])
        shard_name = shard_info["shard"]
        shard_source = get_source_name(shard_info)

        tokens = np.fromfile(shard_path, dtype=SHARD_DTYPE).astype(np.int64)
        dataset = ShardDataset(tokens, BLOCK_SIZE, STRIDE)
        if len(dataset) == 0:
            continue

        loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True,
                            num_workers=2, pin_memory=True)
        pbar = tqdm(loader,
                    desc=f"Ep {epoch} [{shard_idx+1}/{len(epoch_shards)}] {shard_source}/{shard_name}",
                    dynamic_ncols=True, mininterval=1.0)

        for batch_idx, (xb, yb) in enumerate(pbar):
            lr = get_lr(curr_step, total_steps)
            for pg in optimizer.param_groups: pg['lr'] = lr

            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            with torch.amp.autocast("cuda", dtype=torch.bfloat16):
                logits = model(xb)
                loss = F.cross_entropy(logits.view(-1, VOCAB_SIZE), yb.view(-1))
                loss = loss / GRAD_ACCUM_STEPS
            loss.backward()

            if (batch_idx + 1) % GRAD_ACCUM_STEPS == 0:
                grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                last_grad_norm = float(grad_norm)
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                curr_step += 1
                final_loss = loss.item() * GRAD_ACCUM_STEPS
                elapsed = time.time() - train_start

                pbar.set_postfix(loss=f"{final_loss:.3f}",
                                 gnorm=f"{last_grad_norm:.2f}",
                                 lr=f"{lr:.2e}",
                                 step=curr_step,
                                 hrs=f"{elapsed/3600:.1f}",
                                 eval=("-" if last_eval_loss is None else f"{last_eval_loss:.3f}"))

                if curr_step % EVAL_EVERY_STEPS == 0:
                    last_eval_loss = run_eval(model, eval_shards, EVAL_BATCHES)
                    eval_history.append({"step": curr_step, "loss": last_eval_loss,
                                         "elapsed_hrs": round(elapsed/3600, 2)})
                    print(f"  [eval @ step {curr_step}] loss={last_eval_loss:.4f} "
                          f"(train={final_loss:.4f}, grad_norm={last_grad_norm:.2f})")

        del tokens, dataset, loader

        if (shard_idx + 1) % SAVE_EVERY_N_SHARDS == 0:
            elapsed = time.time() - train_start
            print(f"  Mid-epoch save at shard {shard_idx+1}/{len(epoch_shards)} "
                  f"(step {curr_step}, loss {final_loss:.3f}, {elapsed/3600:.1f}h elapsed)...")
            torch.save(model.state_dict(), CHECKPOINT_PATH)

    print(f"Saving Epoch {epoch}...")
    torch.save(model.state_dict(), CHECKPOINT_PATH)

# ==================== 10. FINALIZE ====================
total_time = time.time() - train_start
print(f"Training Complete. {total_time/3600:.1f} hours, {curr_step} steps.")
torch.save(model.state_dict(), CHECKPOINT_PATH)

# ==================== 11. SAVE MANIFEST ====================
manifest = {
    "stage": "training",
    "created": datetime.datetime.now().isoformat(),
    "checkpoint": file_info(CHECKPOINT_PATH),
    "tokenization_id": current_tok_id,
    "config": {
        "arch": "synapse_gpt_2b_d2560_l28",
        "block_size": BLOCK_SIZE, "stride": STRIDE,
        "embed_dim": EMBED_DIM, "num_layers": NUM_LAYERS,
        "num_heads": NUM_HEADS, "num_kv_heads": NUM_KV_HEADS,
        "ff_hidden_dim": FF_HIDDEN_DIM, "dropout": DROPOUT,
        "rope_base": ROPE_BASE, "rmsnorm_eps": RMSNORM_EPS,
        "gradient_checkpointing": GRAD_CHECKPOINT,
        "vocab_size_raw": VOCAB_SIZE_RAW, "vocab_size_padded": VOCAB_SIZE,
        "shard_dtype": str(SHARD_DTYPE),
        "n_params": n_params, "n_trainable": n_trainable,
        "batch_size": BATCH_SIZE, "grad_accum_steps": GRAD_ACCUM_STEPS,
        "epochs": EPOCHS, "max_lr": MAX_LR, "min_lr": MIN_LR,
        "warmup_steps": WARMUP_STEPS, "weight_decay": WEIGHT_DECAY,
        "betas": list(BETAS), "grad_clip": GRAD_CLIP,
        "precision": "bfloat16", "optimizer": "AdamW (fused)",
        "lr_schedule": "cosine_with_warmup",
    },
    "data_selection": {
        "max_tokens": MAX_TOKENS,
        "data_mix": DATA_MIX,
        "selected_shards": len(selected_shards),
        "selected_tokens": selected_tokens,
        "sources": dict(selected_by_source),
        "eval_shards": len(eval_shards),
        "eval_tokens": sum(s["tokens"] for s in eval_shards),
    },
    "results": {
        "final_loss": final_loss,
        "final_eval_loss": last_eval_loss,
        "eval_history": eval_history,
        "last_grad_norm": last_grad_norm,
        "total_optimizer_steps": curr_step,
        "total_hours": round(total_time / 3600, 2),
        "resumed_from": resume_path,
    },
}
manifest_path = os.path.join(MANIFEST_DIR, "training_latest.json")
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)
print(f"Manifest saved: {manifest_path}")
print(f"  tokenization_id: {current_tok_id}")
print(f"  final_loss: {final_loss}")
print(f"  final_eval_loss: {last_eval_loss}")
print(f"  total_steps: {curr_step}")